# CESR planar wiggler: from a four-potential to a transport map

This notebook derives the continuous-field model used for the two periodic planar wigglers in the CESR lattice. It then constructs the corresponding SciBmad element and obtains its linear transport matrix by propagating a GTPSA map through the same symplectic tracking kernel used for particles.

The model is parameterized as

$$
(B_{\max},L_{\mathrm{period}},N_{\mathrm{period}})
\longrightarrow A^\mu(x,y,s,t)
\longrightarrow H(\mathbf q,\mathbf p;s)
\longrightarrow \left(\partial_{\mathbf p}H,-\partial_{\mathbf q}H\right)
\xrightarrow{\text{implicit Yoshida}} \mathcal M
\longrightarrow (c,R,T,\ldots).
$$

A Bmad or Tao map is therefore a validation target, not a frozen replacement for the element model.

## 1. Periodic planar-wiggler field

Bmad's periodic planar model can be written as

$$
\begin{aligned}
B_x &=-B_{\max}\frac{k_x}{k_y}\sin(k_xx)\sinh(k_yy)\cos\theta,\\
B_y &= B_{\max}\cos(k_xx)\cosh(k_yy)\cos\theta,\\
B_s &=-B_{\max}\frac{k_s}{k_y}\cos(k_xx)\sinh(k_yy)\sin\theta,
\end{aligned}
$$

with

$$
k_y^2=k_x^2+k_s^2,\qquad k_s=\frac{2\pi}{L_{\mathrm{period}}},\qquad \theta=k_ss+\phi_s.
$$

The CESR file supplies only B_MAX, L_PERIOD, and N_PERIOD, so the corresponding specialization is $k_x=0$ and $k_y=k_s\equiv k_w$:

$$
B_x=0,\qquad B_y=B_{\max}\cosh(k_wy)\cos\theta,\qquad B_s=-B_{\max}\sinh(k_wy)\sin\theta.
$$

The phase $\phi_s=-k_wL/2$ makes the on-axis vertical field symmetric about the element center.

## 2. Constructing the four-potential

For a magnetostatic element, $\phi=0$ and $\mathbf B=\nabla\times\mathbf A$. Gauge freedom allows the particularly simple choice

$$
A^\mu=(\phi,A_x,A_y,A_s),\qquad \phi=A_y=A_s=0,
$$

$$
A_x=\frac{B_{\max}}{k_w}\cosh(k_wy)\sin(k_ws+\phi_s).
$$

Taking the curl gives

$$
B_y=\frac{\partial A_x}{\partial s}=B_{\max}\cosh(k_wy)\cos\theta,
$$

$$
B_s=-\frac{\partial A_x}{\partial y}=-B_{\max}\sinh(k_wy)\sin\theta.
$$

The Julia callback returns the four potential and all sixteen derivatives $\partial A_\alpha/\partial(x,y,s,t)$. Supplying the derivatives analytically avoids nested numerical differentiation inside the implicit Hamiltonian integrator.

## 3. From the four-potential to the $s$-Hamiltonian

Electromagnetic minimal coupling replaces mechanical momentum by canonical momentum minus the normalized vector potential:

$$
\pi_x=p_x-a_x,\qquad \pi_y=p_y-a_y,\qquad a_i=\frac{A_i}{p_0/q}.
$$

Here $p_x,p_y$ are canonical momenta and $\pi_x,\pi_y$ are mechanical momenta normalized by the reference momentum. BeamTracking first converts the public Bmad coordinates to its internal normalized canonical coordinates. In those coordinates its general static-field $s$-Hamiltonian is, up to an irrelevant additive constant,

$$
H=-(1+gx)p_s-a_s+\frac{p_z}{\beta_0},
$$

with

$$
p_s=\sqrt{\left(p_z+\frac{1}{\beta_0}-\varphi\right)^2-\widetilde m^2-(p_x-a_x)^2-(p_y-a_y)^2},\qquad \widetilde m=\frac{1}{\beta_0\gamma_0}.
$$

The CESR wiggler is straight and magnetostatic, so $g=0$, $\varphi=0$, and $a_y=a_s=0$. Its Hamiltonian therefore reduces to

$$
H=-\sqrt{\left(p_z+\frac{1}{\beta_0}\right)^2-\widetilde m^2-(p_x-a_x(y,s))^2-p_y^2}+\frac{p_z}{\beta_0}.
$$

For example,

$$
\frac{dx}{ds}=\frac{\partial H}{\partial p_x}=\frac{p_x-a_x}{p_s},\qquad \frac{dp_y}{ds}=-\frac{\partial H}{\partial y}=\frac{(p_x-a_x)\,\partial a_x/\partial y}{p_s}.
$$

Thus canonical $p_x=0$ does not imply zero horizontal mechanical motion: inside the wiggler, $\pi_x=-a_x(s)$ generally is nonzero. The explicit $s$ dependence of $a_x$ produces the fast horizontal oscillation, while its $y$ dependence produces vertical focusing.

## 4. Hamilton's equations and the implicit Yoshida map

Once $H(\mathbf q,\mathbf p;s)$ is fixed, the radiation-free trajectory is determined by

$$
\frac{d\mathbf q}{ds}=\frac{\partial H}{\partial\mathbf p},\qquad \frac{d\mathbf p}{ds}=-\frac{\partial H}{\partial\mathbf q}.
$$

Because $a_x(\mathbf q,s)$ appears inside $(p_x-a_x)^2$, this Hamiltonian is not separable into a pure $T(\mathbf p)+V(\mathbf q)$ drift-kick form. BeamTracking therefore uses an implicit symmetric symplectic base step: it evaluates the potential and its Jacobian at intermediate states and solves the coupled position and momentum updates.

A sixth-order Yoshida step is a symmetric composition of seven such second-order implicit maps,

$$
S_6(h)=S_2(w_3h)S_2(w_2h)S_2(w_1h)S_2(w_0h)S_2(w_1h)S_2(w_2h)S_2(w_3h).
$$

In the public SciBmad API this element does **not** pass a user-written `H` function directly to `Yoshida`. Instead, `LineElement` stores the four-potential callback, its parameters, and the Yoshida method. At every substep the generic BeamTracking four-potential kernel normalizes $A^\mu$, constructs $\partial H/\partial\mathbf q$ and $\partial H/\partial\mathbf p$, performs the implicit solve, and composes the result with the Yoshida coefficients. With 16 slices per period and 12 periods, the CESR element has 192 top-level steps; each sixth-order step contains seven implicit substeps.

Radiation damping and fluctuations are separate non-Hamiltonian additions. They are disabled in the default CESR wiggler, so the map discussed here is symplectic.

## Transport-map target and GTPSA

Let $\mathbf z=(x,p_x,y,p_y,z,p_z)^T$ denote canonical phase-space coordinates. Near a reference point $\mathbf z_0$, the tracked map has the expansion

$$
z_i^{\mathrm{out}}=c_i+\sum_jR_{ij}\Delta z_j+\frac12\sum_{jk}T_{ijk}\Delta z_j\Delta z_k+\cdots,
$$

where $\Delta\mathbf z=\mathbf z-\mathbf z_0$ and

$$
c=\mathcal M(\mathbf z_0),\qquad R_{ij}=\left.\frac{\partial\mathcal M_i}{\partial z_j}\right|_{\mathbf z_0}.
$$

GTPSA represents each input coordinate as a truncated polynomial, $z_i=z_{0,i}+\delta_i$. Tracking these six polynomials once propagates every retained coefficient through the element. The Jacobian of the resulting DAMap is therefore $R$ directly; no finite-difference step size is introduced.

For radiation-free Hamiltonian tracking, the first-order matrix should satisfy

$$R^TJR=J,\qquad J=\operatorname{diag}(J_2,J_2,J_2),\qquad J_2=\begin{bmatrix}0&1\\-1&0\end{bmatrix}. $$

In [ ]:
using SciBmad
using GTPSA
using LinearAlgebra
using CairoMakie

cesr_dir = normpath(joinpath(@__DIR__, "..", "..", "wigglers"))
include(joinpath(cesr_dir, "wiggler.jl"))
using .WigglerModels

In [ ]:
const CESR_B_MAX = 1.17               # T
const CESR_L_PERIOD = 0.19625         # m
const CESR_N_PERIOD = 12
const CESR_L = CESR_N_PERIOD * CESR_L_PERIOD
const CESR_P0C = 5.2889999753148e9    # eV/c

scales = planar_wiggler_scales(
    B_max=CESR_B_MAX,
    L_period=CESR_L_PERIOD,
    p0c=CESR_P0C,
)

The basic scales provide useful checks:

$$B\rho=17.6422\;\mathrm{T\,m},\qquad \rho_w=\frac{B\rho}{B_{\max}}=15.0788\;\mathrm m,$$

$$a_x=\frac{1}{\rho_wk_w^2}=64.698\;\mu\mathrm m,\qquad a_{x'}=\frac{1}{\rho_wk_w}=2.071\;\mathrm{mrad}.$$

The horizontal amplitude reproduces OSC_AMPLITUDE in the source CESR lattice. The leading averaged vertical focusing is

$$\langle K_y\rangle\simeq\frac{1}{2\rho_w^2}=2.1991\times10^{-3}\;\mathrm{m}^{-2}. $$

In [ ]:
k_w = 2pi / CESR_L_PERIOD
phase = -k_w * CESR_L / 2
field_params = (CESR_B_MAX, k_w, phase)

s_grid = range(0, CESR_L; length=1201)
B_y_axis = [planar_wiggler_field(0.0, 0.0, s, field_params)[2] for s in s_grid]

fig = Figure(size=(900, 360))
ax = Axis(fig[1, 1], xlabel="local s [m]", ylabel="B_y on axis [T]",
          title="CESR planar-wiggler on-axis field")
lines!(ax, s_grid, B_y_axis, linewidth=2)
fig

In [ ]:
x_test, y_test, s_test, t_test = 1.0e-3, 2.0e-3, 0.37, 0.0
potential, dA = planar_wiggler_four_potential(x_test, y_test, s_test, t_test, field_params)

# With Ay = As = 0, By = dAx/ds and Bs = -dAx/dy.
B_from_curl = (0.0, dA[7], -dA[6])
B_direct = planar_wiggler_field(x_test, y_test, s_test, field_params)

(potential=potential, B_from_curl=B_from_curl, B_direct=B_direct,
 max_error=maximum(abs.(B_from_curl .- B_direct)))

The next cell writes the straight, magnetostatic Hamiltonian explicitly as a diagnostic. It mirrors the mathematical structure used inside BeamTracking, but it is not a second tracking implementation. The production tracker obtains the same normalized potentials and Hamiltonian derivatives from the `four_potential` callback.

In [ ]:
const ELECTRON_MC2 = 0.510998950e6  # eV
beta_gamma_0 = CESR_P0C / ELECTRON_MC2
beta_0 = beta_gamma_0 / sqrt(1 + beta_gamma_0^2)
m_tilde = inv(beta_gamma_0)

# Educational reconstruction of the straight, magnetostatic s-Hamiltonian.
# The electron has q < 0, so its signed p0/q is -B_rho.
signed_p_over_q_ref = -scales.B_rho
function wiggler_s_hamiltonian(v, s; field_params=field_params)
    x, px, y, py, z, pz = v
    potential, dA = planar_wiggler_four_potential(x, y, s, 0.0, field_params)
    ax = potential[2] / signed_p_over_q_ref
    dax_dy = dA[6] / signed_p_over_q_ref
    pi_x = px - ax
    relative_p = pz + inv(beta_0)
    p_s = sqrt(relative_p^2 - m_tilde^2 - pi_x^2 - py^2)
    H = -p_s + pz / beta_0

    dH_dp = (pi_x / p_s, py / p_s, -relative_p / p_s + inv(beta_0))
    dH_dq = (zero(H), -pi_x * dax_dy / p_s, zero(H))
    return (; H, dH_dq, dH_dp, ax, pi_x, p_s)
end

diagnostic_state = zeros(6)
diagnostic_s = CESR_L_PERIOD / 4
hamiltonian_diagnostic = wiggler_s_hamiltonian(diagnostic_state, diagnostic_s)

# Hamilton's equations: dq/ds = dH/dp and dp/ds = -dH/dq.
hamiltonian_flow = (
    dq_ds=hamiltonian_diagnostic.dH_dp,
    dp_ds=.-hamiltonian_diagnostic.dH_dq,
)
(; hamiltonian_diagnostic, hamiltonian_flow)

## 5. Construct and track the SciBmad element

`PlanarWiggler` returns a normal Beamlines `LineElement` with `FourPotentialParams` and a sixth-order Yoshida integration method. This is the actual connection between the potential and the integrator: `four_potential` supplies $A^\mu$ and its Jacobian, while `tracking_method` selects the implicit Yoshida composition. The physical inputs remain the Bmad field and period parameters. With sixteen slices per period, each CESR wiggler uses 192 top-level integration steps.

In [ ]:
wiggler = PlanarWiggler(
    alias="WIG_TEST",
    L=CESR_L,
    B_max=CESR_B_MAX,
    L_period=CESR_L_PERIOD,
    N_period=CESR_N_PERIOD,
    slices_per_period=16,
    order=6,
)

wiggler_line = Beamline([wiggler], species_ref=Species("electron"), E_ref=CESR_P0C)
tracked = track(wiggler_line; v0=zeros(1, 6), use_KA=false, use_explicit_SIMD=false)
zeroth_order_particle = vec(tracked.v[1, :, end])

# Public API contract: the generic internal kernel constructs dH/dq and dH/dp.
tracking_contract = (
    four_potential=wiggler.four_potential,
    four_potential_params=wiggler.four_potential_params,
    physical_potential=!wiggler.four_potential_normalized,
    integrator=wiggler.tracking_method,
)
(; zeroth_order_particle, tracking_contract)

## 6. Extract the linear map with GTPSA

The helper below creates a first-order Descriptor, forms the six GTPSA variables about the requested reference coordinate, tracks them through the beamline, and returns a DAMap. Calling jacobian on that map reads the transported first-order coefficients.

The small fifth-coordinate constant is physical: a particle oscillating inside the wiggler follows a slightly longer path than the straight reference trajectory.

In [ ]:
wiggler_map = gtpsa_transport_map(wiggler_line; v0=zeros(6), order=1)
R = Matrix(jacobian(wiggler_map))
map_constant = scalar.(wiggler_map.v)

J2 = [0.0 1.0; -1.0 0.0]
J = kron(Matrix{Float64}(I, 3, 3), J2)
symplectic_residual = opnorm(transpose(R) * J * R - J, Inf)

(constant=map_constant, R=R, det_R=det(R),
 symplectic_residual=symplectic_residual)

## 7. Compare with the averaged focusing model

After averaging over the fast oscillation, the horizontal plane is approximately a drift, while the vertical plane is approximately a constant-focusing channel:

$$R_x\simeq\begin{bmatrix}1&L\\0&1\end{bmatrix},$$

$$R_y\simeq\begin{bmatrix}\cos(\sqrt{K_y}L)&\sin(\sqrt{K_y}L)/\sqrt{K_y}\\-\sqrt{K_y}\sin(\sqrt{K_y}L)&\cos(\sqrt{K_y}L)\end{bmatrix}. $$

This approximation is a low-order check only; it does not replace the continuous field.

In [ ]:
K_y = scales.K_y_average
root_K = sqrt(K_y)
R_x_average = [1.0 CESR_L; 0.0 1.0]
R_y_average = [
    cos(root_K * CESR_L)           sin(root_K * CESR_L) / root_K
   -root_K * sin(root_K * CESR_L)  cos(root_K * CESR_L)
]

(horizontal_gtpsa=R[1:2, 1:2], horizontal_average=R_x_average,
 vertical_gtpsa=R[3:4, 3:4], vertical_average=R_y_average)

## 8. Bmad validation and the current full-ring Twiss status

For a Bmad comparison, inspect WIG_W and WIG_E with Tao and compare the constant vector, all entries of the 6 by 6 matrix, path length, off-momentum terms, and the full-ring tunes. Continuous-field Bmad tracking is the closest reference for this four-potential model; bmad_standard uses a specialized planar-wiggler approximation and need not agree term by term.

Three independent issues were separated while diagnosing the current full CESR model:

1. The RF cavities are disabled, so CESR must be treated as a coasting ring. Finite-difference coast detection produces a tiny numerical derivative and can misclassify it as bunched; set coasting_beam=true when diagnosing the closed orbit.
2. BeamTracking's implicit four-potential kernel allocates temporary dynamic TPS objects from GTPSA.desc_current. SciBmad restores the previous global descriptor when it creates a default temporary descriptor, which produces No Descriptor defined. Create a Descriptor explicitly and pass it as GTPSA_descriptor until the upstream allocation is fixed.
3. A four-dimensional coasting closed-orbit Newton solve driven entirely by GTPSA converges in three iterations to a residual below 5e-14. Normal-form analysis then reaches the physical stability check and reports an unstable orbital eigenmode. The no-kicker one-turn GTPSA matrix has transverse eigenvalue magnitudes approximately (10.0540, 1, 1, 0.09946). Replacing both wigglers by drifts still gives approximately (9.2743, 1, 1, 0.10782), so the wiggler model is not the cause of the Twiss failure. The remaining task is an element-by-element Bmad/SciBmad map comparison of the converted baseline, especially combined superimposed IR elements and bend/fringe conventions.